# Prometheus-1B: Phase 2 - Continued Pre-Training (CPT)
**Llama 3.2 1B Base 모델에 다도메인 데이터로 지식 주입**

| 항목 | 설정 |
|------|------|
| Base 모델 | `unsloth/Llama-3.2-1B` (Base, Instruct 아님!) |
| 학습 방식 | Full BF16 (A100 80GB VRAM 활용) |
| Effective Batch | 32 (batch=8 x grad_accum=4) |
| 시퀀스 길이 | 4096 |
| LR | 2e-5, Cosine Decay, Warmup 1000 steps |
| 총 스텝 | ~26,000 (데이터 크기에 따라 자동 계산) |
| 체크포인트 | 500 steps마다 저장, 최근 5개 유지 |

**예상 소요**: 2.5-3일 (A100, 세션 3-4회)  
**핵심**: 세션 재시작 시 `resume_from_checkpoint`으로 자동 이어서 학습

## 1. 환경 설정 및 패키지 설치

In [ ]:
%%capture
# 필수 패키지 설치 (Unsloth 제거 - 순수 transformers로 학습)
!pip install transformers datasets accelerate
!pip install wandb

In [ ]:
import os
import torch
from google.colab import drive, userdata

# Google Drive 마운트
drive.mount('/content/drive')
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

# GPU 확인
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB")
print(f"BF16 지원: {torch.cuda.is_bf16_supported()}")

# Wandb 설정 (선택사항 - API키 없으면 건너뜀)
try:
    import wandb
    wandb_key = userdata.get("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    USE_WANDB = True
    print("Wandb: enabled")
except Exception:
    USE_WANDB = False
    print("Wandb: disabled (API key not found, training will still work)")

In [ ]:
# Colab 세션 끊김 방지 (백그라운드 스레드)
import threading
import time
import IPython

def keep_alive():
    while True:
        time.sleep(120)  # 2분마다
        IPython.display.clear_output(wait=True)

keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("Keep-alive thread started (2min interval)")

## 2. 모델 로드
**중요**: Instruct가 아닌 **Base 모델**을 사용합니다. Instruct 모델에 CPT를 하면 지시 수행 능력이 파괴됩니다(Catastrophic Forgetting).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# === 핵심 설정 ===
MODEL_NAME = "meta-llama/Llama-3.2-1B"  # BASE 모델! (Instruct 아님!)
MAX_SEQ_LENGTH = 4096
OUTPUT_DIR = "/content/drive/MyDrive/Prometheus-1B-CPT"
DATA_DIR = "/content/drive/MyDrive/Prometheus-1B-Data"

# 모델 로드 (Full BF16 - A100 80GB이므로 양자화 불필요)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token,
    attn_implementation="eager",  # gradient checkpointing 호환
)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. 데이터 로드
Phase 1에서 준비한 사전 토큰화된 데이터를 로드합니다.

In [ ]:
from datasets import load_from_disk

# 사전 토큰화된 데이터 로드
train_dataset = load_from_disk(os.path.join(DATA_DIR, "train"))
eval_dataset = load_from_disk(os.path.join(DATA_DIR, "eval"))

print(f"Train: {len(train_dataset):,} sequences ({len(train_dataset) * MAX_SEQ_LENGTH / 1e9:.2f}B tokens)")
print(f"Eval:  {len(eval_dataset):,} sequences")
print(f"Features: {train_dataset.features}")

# 학습 스텝 수 자동 계산
BATCH_SIZE = 8
GRAD_ACCUM = 4
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM  # 32
total_steps = len(train_dataset) // EFFECTIVE_BATCH
print(f"\nEffective batch size: {EFFECTIVE_BATCH}")
print(f"Total training steps: {total_steps:,}")
print(f"Estimated time on A100: {total_steps * EFFECTIVE_BATCH * MAX_SEQ_LENGTH / 20000 / 3600:.1f} hours")

## 4. 학습 설정 (Training Arguments)
CPT에 최적화된 하이퍼파라미터입니다.

**GUIDE.md와의 차이점:**
- SFTTrainer → `Trainer` + `DataCollatorForLanguageModeling` (CPT에 적합)
- LR 5e-5 → `2e-5` (기존 지식 보존)
- 스케줄 없음 → `Cosine Decay + Warmup`
- 평가 없음 → `1000 steps마다 eval`

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    # === 출력 및 체크포인트 ===
    output_dir=OUTPUT_DIR,
    
    # === 배치 및 메모리 ===
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # === 학습률 (CPT 최적화) ===
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=1000,
    weight_decay=0.1,
    max_grad_norm=1.0,
    
    # === 정밀도 ===
    bf16=True,
    
    # === 체크포인트 ===
    save_steps=500,
    save_total_limit=5,
    
    # === 학습 범위 ===
    num_train_epochs=1,
    
    # === 로깅 및 평가 ===
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=1000,
    
    # === 성능 최적화 ===
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    gradient_checkpointing=True,
    
    # === 모니터링 ===
    report_to="wandb" if USE_WANDB else "none",
    run_name="prometheus-1b-cpt" if USE_WANDB else None,
)

print("Training Arguments configured!")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Scheduler: {training_args.lr_scheduler_type}")
print(f"  Warmup: {training_args.warmup_steps} steps")
print(f"  Save every: {training_args.save_steps} steps")
print(f"  Eval every: {training_args.eval_steps} steps")

## 5. Trainer 생성 및 학습 시작
이 셀이 메인 학습 루프입니다. 세션이 끊기면 이 노트북을 처음부터 다시 실행하세요 - `resume_from_checkpoint`이 마지막 체크포인트부터 자동으로 이어서 학습합니다.

> **A100 예상 속도**: ~20,000-24,000 tokens/sec → 약 2.5-3일 소요

In [ ]:
# Gradient Checkpointing 활성화 (VRAM 절약)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Data Collator (CLM = Causal Language Modeling)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # CLM (autoregressive), not MLM
)

# Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

# 체크포인트 자동 감지 및 학습 시작
checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"Resuming from checkpoint: {checkpoint}")
    else:
        print("No checkpoint found. Starting fresh training.")
else:
    print("Starting fresh training.")

# 학습 시작!
print(f"\n{'='*60}")
print("TRAINING START")
print(f"{'='*60}")
trainer.train(resume_from_checkpoint=checkpoint)

## 6. 학습 완료 후 최종 모델 저장
학습이 완료되면 최종 모델을 저장합니다. 이 모델이 Phase 3 (SFT)의 입력이 됩니다.

In [ ]:
# 최종 모델 저장
FINAL_MODEL_DIR = "/content/drive/MyDrive/Prometheus-1B-CPT-Final"

print("Saving final CPT model...")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"Final model saved to: {FINAL_MODEL_DIR}")
print(f"\nTraining complete! Proceed to Phase 3 (SFT) with notebook 03_train_sft.ipynb")

# 학습 로그 요약
if trainer.state.log_history:
    train_losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]
    eval_losses = [x["eval_loss"] for x in trainer.state.log_history if "eval_loss" in x]
    
    if train_losses:
        print(f"\nTrain loss: {train_losses[0]:.4f} (start) → {train_losses[-1]:.4f} (end)")
    if eval_losses:
        print(f"Eval loss:  {eval_losses[0]:.4f} (start) → {eval_losses[-1]:.4f} (end)")